# PyTorch Knowledge Graph Analysis

This notebook provides detailed analysis of the enhanced PyTorch knowledge graph with edge relationships.

In [1]:
# Load the enhanced knowledge graph
from visualize_knowledge_graph import load_knowledge_graph, build_nx_graph

# Load the enhanced graph
kg = load_knowledge_graph('enhanced_knowledge_graph_with_edges.json')
G = build_nx_graph(kg)

print(f"Loaded enhanced graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

Loaded enhanced graph with 45883 nodes and 21748 edges


## Enhanced Graph Analysis

In [2]:
# Node type distribution
from collections import Counter
node_type_counts = Counter(node_data.get('type', 'other') for _, node_data in G.nodes(data=True))
print("Node type distribution:")
for node_type, count in sorted(node_type_counts.items(), key=lambda x: -x[1]):
    print(f"  {node_type}: {count}")

# Edge type distribution
edge_type_counts = Counter(data.get('link_type', 'other') for _, _, data in G.edges(data=True))
print("\nEdge type distribution:")
for edge_type, count in sorted(edge_type_counts.items(), key=lambda x: -x[1]):
    print(f"  {edge_type}: {count}")

Node type distribution:
  function: 35517
  class: 5202
  other: 5074
  module: 90

Edge type distribution:
  imports: 18909
  inheritance: 2781
  calls: 35
  inherits: 23


## Enhanced Linear Analysis

In [3]:
# Detailed analysis of torch.nn.Linear
print("=== Detailed Analysis of torch.nn.Linear ===")

linear_node_id = 'torch/nn/modules/linear.py:Linear'

if linear_node_id in G.nodes():
    linear_node_data = G.nodes[linear_node_id]
    print(f"Linear node details:")
    print(f"  Name: {linear_node_data.get('name')}")
    print(f"  Type: {linear_node_data.get('type')}")
    print(f"  Path: {linear_node_data.get('path')}")
    
    # Get all neighbors
    predecessors = list(G.predecessors(linear_node_id))
    successors = list(G.successors(linear_node_id))
    
    print(f"\nRelationships:")
    print(f"  Incoming relationships: {len(predecessors)}")
    print(f"  Outgoing relationships: {len(successors)}")
    
    # Show outgoing relationships (the key ones)
    if successors:
        print("\nOutgoing relationships (inheritance):")
        for succ in successors:
            succ_data = G.nodes[succ]
            edge_data = G.get_edge_data(linear_node_id, succ)
            edge_type = edge_data.get('link_type', 'unknown') if edge_data else 'unknown'
            print(f"  {succ_data.get('name', succ)} ({edge_type})")
            
            # Show detailed information for inheritance
            if edge_type == 'inheritance':
                print(f"    Path: {succ_data.get('path')}")
                print(f"    Type: {succ_data.get('type')}")
                print(f"    Description: {succ_data.get('description', 'No description')[:100]}...")
else:
    print("Could not find torch.nn.Linear in the graph")

=== Detailed Analysis of torch.nn.Linear ===
Linear node details:
  Name: Linear
  Type: class
  Path: torch/nn/modules/linear.py

Relationships:
  Incoming relationships: 0
  Outgoing relationships: 1

Outgoing relationships (inheritance):
  Module (inheritance)
    Path: None
    Type: None
    Description: No description...


## Subsystem Analysis

In [4]:
# Analyze key subsystems
print("=== Key Subsystem Analysis ===")

# Find various Linear-related classes
linear_classes = [n for n, d in G.nodes(data=True) if 'Linear' in d.get('name', '') and d.get('type') == 'class']
print(f"Found {len(linear_classes)} Linear-related classes:")
for node in sorted(linear_classes, key=lambda x: G.nodes[x].get('name', ''))[:10]:
    name = G.nodes[node].get('name', node)
    path = G.nodes[node].get('path', '')
    print(f"  {name} in {path}")
    
if len(linear_classes) > 10:
    print(f"  ... and {len(linear_classes) - 10} more")

=== Key Subsystem Analysis ===
Found 84 Linear-related classes:
  AnnotatedSingleLayerLinearModel in torch/testing/_internal/common_quantization.py
  AnnotatedTwoLayerLinearModel in torch/testing/_internal/common_quantization.py
  BatchLinearLHSFusion in torch/_inductor/fx_passes/group_batch_fusion.py
  Conv2dWithTwoLinear in torch/testing/_internal/common_quantization.py
  Conv2dWithTwoLinearPermute in torch/testing/_internal/common_quantization.py
  ConvBnReLU2dAndLinearReLU in torch/testing/_internal/common_quantization.py
  ConvLinearWPermute in torch/testing/_internal/common_quantization.py
  DeFusedEmbeddingBagLinear in torch/testing/_internal/common_quantization.py
  DoubleLinear in torch/testing/_internal/common_fsdp.py
  EmbeddingConvLinearModule in torch/testing/_internal/common_quantization.py
  ... and 74 more


## Edge Relationship Analysis

In [5]:
# Analyze specific edge relationships
print("=== Edge Relationship Analysis ===")

# Show inheritance relationships
inheritance_edges = [(u, v, data) for u, v, data in G.edges(data=True) if data.get('link_type') == 'inheritance']
print(f"Found {len(inheritance_edges)} inheritance relationships")

# Show first few inheritance relationships
for i, (u, v, data) in enumerate(inheritance_edges[:5]):
    u_name = G.nodes[u].get('name', u)
    v_name = G.nodes[v].get('name', v)
    print(f"  {u_name} inherits from {v_name}")

# Show import relationships
import_edges = [(u, v, data) for u, v, data in G.edges(data=True) if data.get('link_type') == 'imports']
print(f"\nFound {len(import_edges)} import relationships")

# Show first few import relationships
for i, (u, v, data) in enumerate(import_edges[:5]):
    u_name = G.nodes[u].get('name', u)
    v_name = G.nodes[v].get('name', v)
    print(f"  {u_name} imports {v_name}")

=== Edge Relationship Analysis ===
Found 2781 inheritance relationships
  ByteStorage inherits from _LegacyStorage
  DoubleStorage inherits from _LegacyStorage
  FloatStorage inherits from _LegacyStorage
  HalfStorage inherits from _LegacyStorage
  LongStorage inherits from _LegacyStorage

Found 18909 import relationships
  torch imports _C
  torch imports _ops
  torch imports torch._dynamo
  torch imports torch._inductor
  torch imports torch.fx


## Conclusion

The enhanced knowledge graph provides significant improvements over the original:
- 45,883 nodes vs 128 nodes
- 21,748 edges vs 120 edges
- Rich edge relationships showing inheritance, imports, and usage
- Detailed analysis of torch.nn.Linear and its relationships
- Comprehensive exploration of PyTorch subsystems